# Porosity Analysis Pipeline

Parameterised notebook — run via papermill or interactively.

In [ ]:
# ── Papermill parameters ─────────────────────────────────────────────────────
INPUT_TIF  = 'volume.tif'       # overridden by papermill
OUTPUT_DIR = '.'                 # overridden by papermill
REPO_ROOT  = '/home/jorgecabrejas/Dev/GenAI'  # overridden by papermill
VOXEL_UM   = 25.0
RENDER_GIF = True   # set False to skip Step 8 cinematic animation

# Sauvola / detection parameters
SAUVOLA_RADIUS     = 31
SAUVOLA_K          = 0.125
MATERIAL_THRESHOLD = 10
N_EROSIONS         = 0

# Size filter
MIN_VOXELS = 8
MAX_VOXELS = None   # set to int to cap large pores

# Sphericity (slow for >10k pores)
COMPUTE_SPHERICITY = True

# Spatial metrics sampling budget (reduce for speed)
N_SAMPLES_S2 = 200_000
N_SAMPLES_L  =  50_000
N_RAYS       =   5_000

In [ ]:
import json, math, shutil, subprocess, sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from IPython.display import Image as IPImage, display
from scipy import ndimage
from scipy.spatial import KDTree
from scipy.stats import lognorm, wasserstein_distance
from skimage import measure, morphology, filters
from skimage.filters import threshold_sauvola
from skimage.measure import marching_cubes, mesh_surface_area, euler_number
from tqdm.auto import tqdm

# Make repo root and inference/ importable
_repo = Path(REPO_ROOT)
sys.path.insert(0, str(_repo))
sys.path.insert(0, str(_repo / 'inference'))

from onlypores import onlypores, clean_pores
from analysis_utils_core import (
    size_filter_pores, compute_regionprops_df, compute_vvf_map,
    compute_z_profile, write_statistics_json, export_artifacts,
    plot_pore_size_histograms, plot_regionprops_distributions,
)
from analysis_utils_spatial import spatial_metrics
from analysis_utils_render import render_presentation

INPUT_TIF  = Path(INPUT_TIF)
OUTPUT_DIR = Path(OUTPUT_DIR)
REPO_ROOT  = Path(REPO_ROOT)
print(f"INPUT_TIF : {INPUT_TIF}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

## Step 1 — Load & Convert

In [ ]:
raw = tifffile.imread(INPUT_TIF)
print(f"Raw shape={raw.shape}  dtype={raw.dtype}  min={raw.min():.4f}  max={raw.max():.4f}")

vol = (raw * 255).clip(0, 255).astype(np.uint8)
print(f"uint8 shape={vol.shape}  min={vol.min()}  max={vol.max()}")

assert not vol[0].any(), "Slice 0 should be all zeros — check volume format!"
print("Slice 0 all-zero: ✓")
print(f"Slice 1 non-zero voxels: {vol[1].astype(bool).sum():,}")

## Step 2 — Interactive Sauvola Parameter Explorer (2D)

In [ ]:
from ipywidgets import interact, IntSlider, FloatSlider

def _preview(mat_threshold=10, radius=31, k=0.125, z=None):
    if z is None:
        z = vol.shape[0] // 2
    slc = vol[z]                                                     # 2D (Y, X)
    mat_2d = ndimage.binary_fill_holes(slc > mat_threshold)         # 2D material mask
    thresh = threshold_sauvola(slc, window_size=radius, k=k, r=128)
    pore_2d = (~(slc > thresh)) & mat_2d

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    # Left: raw slice with overlays
    axes[0].imshow(slc, cmap='gray', vmin=0, vmax=255)
    from skimage.measure import find_contours
    for c in find_contours(mat_2d.astype(float), 0.5):
        axes[0].plot(c[:, 1], c[:, 0], 'r-', lw=0.8, label='Material')
    for c in find_contours(pore_2d.astype(float), 0.5):
        axes[0].plot(c[:, 1], c[:, 0], 'c-', lw=0.8, label='Pores')
    axes[0].set_title(f'Slice z={z} — raw + overlays')
    axes[0].axis('off')
    # Right: pore binary
    axes[1].imshow(pore_2d, cmap='binary_r')
    axes[1].set_title(f'Pore mask (radius={radius}, k={k:.3f})')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    plt.close(fig)

interact(
    _preview,
    mat_threshold=IntSlider(min=0, max=255, step=1, value=10, description='mat_thresh'),
    radius=IntSlider(min=11, max=101, step=2, value=31, description='radius'),
    k=FloatSlider(min=0.02, max=0.5, step=0.005, value=0.125, description='k'),
    z=IntSlider(min=1, max=vol.shape[0]-1, step=1, value=vol.shape[0]//2, description='z-slice'),
)

## Step 3 — 3D Sauvola Pore Detection

In [ ]:
pore_mask_raw, _sm, _binary = onlypores(
    vol,
    sauvola_radius=SAUVOLA_RADIUS,
    sauvola_k=SAUVOLA_K,
    mask_threshold=MATERIAL_THRESHOLD,
    border_erosion=N_EROSIONS,
)

# Authoritative material mask — vol > 1 correctly excludes slice 0 (all zeros)
material_mask = vol > 1
pore_mask_raw = pore_mask_raw & material_mask

material_voxels = int(material_mask.sum())
raw_pore_voxels = int(pore_mask_raw.sum())
raw_vvf = raw_pore_voxels / material_voxels if material_voxels > 0 else 0.0
print(f"Material voxels    : {material_voxels:,}")
print(f"Raw pore voxels    : {raw_pore_voxels:,}")
print(f"material_mask[0].any() = {bool(material_mask[0].any())}  (must be False)")
print(f"Raw VVF            : {raw_vvf:.4%}")

# Orthogonal overlays
mid_z = vol.shape[0] // 2
mid_y = vol.shape[1] // 2
mid_x = vol.shape[2] // 2
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (slc_v, slc_m, slc_p, title) in zip(axes, [
    (vol[mid_z],    material_mask[mid_z],    pore_mask_raw[mid_z],    f'Z={mid_z}'),
    (vol[:, mid_y], material_mask[:, mid_y], pore_mask_raw[:, mid_y], f'Y={mid_y}'),
    (vol[:, :, mid_x], material_mask[:, :, mid_x], pore_mask_raw[:, :, mid_x], f'X={mid_x}'),
]):
    ax.imshow(slc_v, cmap='gray', vmin=0, vmax=255)
    from skimage.measure import find_contours
    for c in find_contours(slc_m.astype(float), 0.5):
        ax.plot(c[:,1], c[:,0], 'r-', lw=0.6)
    for c in find_contours(slc_p.astype(float), 0.5):
        ax.plot(c[:,1], c[:,0], 'c-', lw=0.6)
    ax.set_title(title); ax.axis('off')
plt.suptitle('Orthogonal Slices — Red: material boundary, Cyan: raw pores')
plt.tight_layout()
display(fig)
plt.close(fig)

## Step 4 — Size Filtering

In [ ]:
# Raw size distribution
pore_labels_raw, n_pores_raw = ndimage.label(pore_mask_raw)
raw_sizes = pd.Series(np.bincount(pore_labels_raw.ravel())[1:], name='voxels')
print(f"Raw pore components: {n_pores_raw:,}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig, ax1, ax2 = plot_pore_size_histograms(raw_sizes, ax_linear=ax1, ax_log=ax2)
plt.suptitle('Raw pore size distribution (before filtering)')
plt.tight_layout(); display(fig); plt.close(fig)

# Apply filter
pore_mask, pore_labels, n_pores, pore_sizes = size_filter_pores(
    pore_mask_raw, min_voxels=MIN_VOXELS, max_voxels=MAX_VOXELS
)
vvf_global = float(pore_mask.sum()) / material_voxels if material_voxels > 0 else 0.0

print(f"\nAfter filtering:")
print(f"  Pores kept : {n_pores:,}")
print(f"  Pore voxels: {int(pore_mask.sum()):,}")
print(f"  Global VVF : {vvf_global:.4%}")

# Overlay cut lines on the histograms
fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(12, 4))
fig2, ax3, ax4 = plot_pore_size_histograms(raw_sizes, ax_linear=ax3, ax_log=ax4)
ax3.axvline(MIN_VOXELS, color='r', linestyle='--', label=f'MIN={MIN_VOXELS}')
ax4.axvline(MIN_VOXELS, color='r', linestyle='--', label=f'MIN={MIN_VOXELS}')
if MAX_VOXELS:
    ax3.axvline(MAX_VOXELS, color='orange', linestyle='--', label=f'MAX={MAX_VOXELS}')
    ax4.axvline(MAX_VOXELS, color='orange', linestyle='--')
ax3.legend(); ax4.legend()
plt.suptitle('Pore size distribution — cut lines')
plt.tight_layout(); display(fig2); plt.close(fig2)

## Step 5 — Regionprops & Pore Characterisation

In [ ]:
df_pores = compute_regionprops_df(pore_labels, voxel_um=VOXEL_UM, compute_sphericity=COMPUTE_SPHERICITY)
print(df_pores.describe())

fig = plot_regionprops_distributions(df_pores, voxel_um=VOXEL_UM)
display(fig); plt.close(fig)

## Step 6 — VVF Map

In [ ]:
vvf_map = compute_vvf_map(pore_mask, material_mask)
print(f"VVF map shape: {vvf_map.shape}")
valid = vvf_map[np.isfinite(vvf_map)]
print(f"VVF map: mean={valid.mean():.4f}  std={valid.std():.4f}  min={valid.min():.4f}  max={valid.max():.4f}")

fig, ax = plt.subplots(figsize=(10, 6))
cmap = plt.cm.inferno.copy(); cmap.set_bad('gray')
im = ax.imshow(vvf_map, cmap=cmap, aspect='auto')
plt.colorbar(im, ax=ax, label='Porosity (VVF)')
ax.set_title('Per-column VVF Map'); ax.set_xlabel('X'); ax.set_ylabel('Y')
plt.tight_layout(); display(fig); plt.close(fig)

## Step 7 — Statistics & Exports

In [ ]:
df_z = compute_z_profile(pore_mask, material_mask)

sauvola_params = {
    'radius': SAUVOLA_RADIUS,
    'k': SAUVOLA_K,
    'material_threshold': MATERIAL_THRESHOLD,
}

stats = write_statistics_json(
    output_path=OUTPUT_DIR / 'statistics.json',
    input_tif=str(INPUT_TIF),
    vol_shape=list(vol.shape),
    voxel_um=VOXEL_UM,
    sauvola_params=sauvola_params,
    material_voxels=material_voxels,
    pore_voxels=int(pore_mask.sum()),
    n_pores=n_pores,
    vvf_global=vvf_global,
    vvf_map=vvf_map,
    df_z=df_z,
    df_pores=df_pores,
)
print(json.dumps({k: v for k, v in stats.items() if not isinstance(v, (list, dict))}, indent=2))

export_artifacts(pore_mask, material_mask, vvf_map, df_z, df_pores, OUTPUT_DIR)
print("\nExports complete:")
for f in ['statistics.json','z_profile.csv','pore_sizes.csv','vvf_map.tif','vvf_map.png',
          'vvf_zones.png','only_pores.tif','sample_mask.tif']:
    p = OUTPUT_DIR / f
    print(f"  {'✓' if p.exists() else '✗'}  {f}")

## Step 8 — Cinematic 3D Presentation Animation

In [ ]:
sample_name = INPUT_TIF.parent.name

if RENDER_GIF:
    result = render_presentation(
        pore_mask=pore_mask,
        material_mask=material_mask,
        vol=vol,
        output_dir=OUTPUT_DIR,
        n_pores=n_pores,
        vvf_global=vvf_global,
        sample_name=sample_name,
    )
    print(f"Backend used : {result['backend']}")
    print(f"GIF          : {result['gif']}")
    print(f"Thumbnail    : {result['thumb']}")
    if result['thumb'].exists():
        display(IPImage(str(result['thumb']), width=640))
else:
    print("RENDER_GIF=False — skipping cinematic animation")

## Step 9 — Spatial Statistics & Paper Metrics

In [ ]:
sm = spatial_metrics(
    pore_mask=pore_mask,
    material_mask=material_mask,
    df_pores=df_pores,
    vvf_global=vvf_global,
    output_dir=OUTPUT_DIR,
    voxel_um=VOXEL_UM,
    n_samples_s2=N_SAMPLES_S2,
    n_samples_l=N_SAMPLES_L,
    n_rays=N_RAYS,
)
print(json.dumps(sm, indent=2, default=str))
if (OUTPUT_DIR / 'spatial_metrics.png').exists():
    display(IPImage(str(OUTPUT_DIR / 'spatial_metrics.png'), width=900))

In [ ]:
print("=" * 50)
print("ANALYSIS SUMMARY")
print("=" * 50)
rows = [
    ('Sample',          sample_name),
    ('Voxel size (µm)', f'{VOXEL_UM}'),
    ('Global VVF',      f'{vvf_global:.4%}'),
    ('Pore count',      f'{n_pores:,}'),
    ('Euler number',    str(sm.get('euler_number', 'N/A'))),
    ('Sv (µm⁻¹)',       f"{sm.get('specific_surface_area_inv_um', 'N/A'):.4f}" if sm.get('specific_surface_area_inv_um') else 'N/A'),
    ('S₂ corr len (µm)',f"{sm.get('s2_correlation_length_um', 'N/A')}"),
]
for k, v in rows:
    print(f"  {k:<25} {v}")